# ClinicASR-IND: Two-Stage Hallucination Guard

This notebook implements the **Hallucination Detection & Flagging** module.

## Architecture
```
Audio Transcript (Whisper output)
         |
         v
[Stage 1] scispaCy BC5CDR NER
  Extracts: Drugs, Symptoms, Dosages, Diagnoses from generated doc
         |
         v
[Stage 2] BiomedBERT + Cross-Encoder Grounding
  Scores each entity: does it appear in the original audio?
         |
         v
Physician Report  GREEN / YELLOW / RED per entity
```

**Why BiomedBERT?** Pre-trained on 29M PubMed abstracts. Outperforms general BERT on medical NER and semantic similarity by ~12% F1 on BioASQ benchmarks.

## Step 1: Install Dependencies

In [ ]:
# ================================================================
# STEP 1 — Environment setup (run once per Colab session)
# ================================================================
# WHY this order matters:
#   thinc (spaCy's tensor backend) compiles Cython extensions that
#   link against numpy's C ABI.  If numpy is upgraded AFTER thinc is
#   already installed you get:
#     ValueError: numpy.dtype size changed, may indicate binary incompatibility
# FIX: pin numpy first, then force-reinstall thinc + spaCy so they
# recompile against the pinned version.
# ================================================================

# 1. Pin numpy to a version whose ABI matches thinc 8.x wheels on Colab
!pip install -q "numpy>=1.24,<2.0"

# 2. Force-reinstall thinc & spaCy so their Cython extensions rebuild
#    against the pinned numpy above
!pip install -q --force-reinstall "thinc>=8.2,<8.4" "spacy>=3.7,<3.9"

# 3. scispaCy + BC5CDR medical NER model
!pip install -q scispacy
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

# 4. Remaining deps (no ABI sensitivity)
!pip install -q transformers accelerate sentence-transformers

# 5. IMPORTANT: restart the runtime after this cell so the freshly
#    compiled thinc/spaCy are picked up cleanly.
#    Runtime > Restart session  (do NOT re-run this cell after restart)
print("\n✅ Install complete. Now go to Runtime > Restart session, then continue from Step 2.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 44.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible

## Step 2: Imports & Configuration

In [1]:
# Quick ABI sanity-check — if this cell fails with the dtype error,
# it means the runtime was not restarted after Step 1.  Restart and retry.
import numpy as np
try:
    import thinc
except ValueError as e:
    raise RuntimeError(
        "numpy/thinc ABI mismatch detected.\n"
        "Please go to Runtime > Restart session and then re-run from this cell."
    ) from e

import torch
import spacy
import json
import re
import numpy as np
from enum import Enum
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import CrossEncoder

# ---- CONFIGURATION ----
CONFIG = {
    "biomedbert_model": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
    "cross_encoder_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    # Traffic-light thresholds
    "verified_threshold": 0.72,   # Above -> GREEN
    "uncertain_threshold": 0.45,  # Between -> YELLOW; below -> RED
    "high_risk_entity_types": ["CHEMICAL", "DISEASE"],
    "chunk_size_sentences": 3,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

print(f"Device: {CONFIG['device']}")
print(f"BiomedBERT: {CONFIG['biomedbert_model']}")

Device: cpu
BiomedBERT: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext


## Step 3: Data Structures

Typed dataclasses for the physician-in-the-loop output.

In [2]:
class ConfidenceLabel(Enum):
    VERIFIED     = "GREEN"    # Grounded in audio transcript
    UNCERTAIN    = "YELLOW"   # Weak grounding - doctor should review
    HALLUCINATED = "RED"      # Not found - likely invented


@dataclass
class ClinicalEntity:
    text: str
    entity_type: str
    source_sentence: str
    confidence_score: float = 0.0
    label: ConfidenceLabel = ConfidenceLabel.HALLUCINATED
    best_matching_transcript_chunk: str = ""
    is_high_risk: bool = False


@dataclass
class HallucinationReport:
    original_transcript: str
    generated_document: str
    entities: List[ClinicalEntity] = field(default_factory=list)
    overall_trust_score: float = 0.0
    flagged_sentences: List[str] = field(default_factory=list)

    def summary(self) -> Dict:
        verified     = sum(1 for e in self.entities if e.label == ConfidenceLabel.VERIFIED)
        uncertain    = sum(1 for e in self.entities if e.label == ConfidenceLabel.UNCERTAIN)
        hallucinated = sum(1 for e in self.entities if e.label == ConfidenceLabel.HALLUCINATED)
        return {
            "total_entities": len(self.entities),
            "verified": verified,
            "uncertain": uncertain,
            "hallucinated": hallucinated,
            "trust_score": round(self.overall_trust_score, 3),
            "high_risk_hallucinations": [
                e.text for e in self.entities
                if e.label == ConfidenceLabel.HALLUCINATED and e.is_high_risk
            ]
        }

print("Data structures defined.")

Data structures defined.


## Step 4 — Stage 1: Medical NER (scispaCy BC5CDR)

The `en_ner_bc5cdr_md` model was trained on the BioCreative V CDR corpus — ~1,500 PubMed articles annotated with **CHEMICAL** and **DISEASE** entities.  

Why not general spaCy?  
- Recognises drug abbreviations (e.g. `HTN`, `DM2`, `HbA1c`)  
- Handles dosage patterns like `500mg bd` or `75mcg OD`  
- ~30% lower false-positive rate on clinical text vs `en_core_web_sm`

In [3]:
# ---------------------------------------------------------------
# FIX: spaCy 3.8 changed include_static_vectors from str to bool.
# We patch the model's config on disk before loading to avoid the
# ConfigValidationError that appears with en_ner_bc5cdr_md 0.5.4.
# ---------------------------------------------------------------
import spacy
import importlib
from pathlib import Path

def patch_bc5cdr_config():
    """Locate the BC5CDR model config and coerce the string 'True'/'False'
    values to proper Python booleans so spaCy 3.8 accepts them."""
    try:
        import en_ner_bc5cdr_md
        model_path = Path(en_ner_bc5cdr_md.__file__).parent
    except Exception:
        # Fallback: let spaCy find it
        model_path = Path(spacy.util.get_package_path('en_ner_bc5cdr_md'))

    cfg_path = model_path / 'config.cfg'
    if not cfg_path.exists():
        print(f'  [WARN] config.cfg not found at {cfg_path}, skipping patch.')
        return

    text = cfg_path.read_text()
    # The broken lines look like: include_static_vectors = True  (string, not bool)
    # spaCy config format uses lowercase true/false for booleans
    # The safest fix is to ensure the value is bare true/false (TOML-style)
    import re
    # Replace  = "True" / = 'True' / = True (string) -> = true
    patched = re.sub(
        r"include_static_vectors\s*=\s*['\"]?True['\"]?",
        "include_static_vectors = true",
        text
    )
    patched = re.sub(
        r"include_static_vectors\s*=\s*['\"]?False['\"]?",
        "include_static_vectors = false",
        patched
    )
    if patched != text:
        cfg_path.write_text(patched)
        print('  [OK] Patched include_static_vectors in config.cfg')
    else:
        print('  [OK] config.cfg already uses correct boolean format')


print('Patching BC5CDR config for spaCy 3.8 compatibility...')
patch_bc5cdr_config()

print('Loading scispaCy BC5CDR model...')
nlp_ner = spacy.load('en_ner_bc5cdr_md')
print(f'NER pipeline components: {nlp_ner.pipe_names}')


def extract_clinical_entities(text: str) -> List[Dict]:
    """
    Stage 1: Run scispaCy NER on the generated medical document.
    Returns list of dicts: entity text, label (CHEMICAL/DISEASE),
    surrounding sentence, and high-risk flag.
    """
    doc = nlp_ner(text)
    entities = []

    for ent in doc.ents:
        # Find the sentence that contains this entity
        containing_sentence = ''
        for sent in doc.sents:
            if ent.start >= sent.start and ent.end <= sent.end:
                containing_sentence = sent.text.strip()
                break

        entities.append({
            'text': ent.text,
            'label': ent.label_,
            'start': ent.start_char,
            'end': ent.end_char,
            'sentence': containing_sentence,
            'is_high_risk': ent.label_ in CONFIG['high_risk_entity_types']
        })

    return entities


# Quick sanity test
test_doc = 'Patient has Diabetes Mellitus Type 2. Prescribed Metformin 500mg twice daily.'
test_ents = extract_clinical_entities(test_doc)
print('\nTest entity extraction:')
for e in test_ents:
    print(f"  [{e['label']}] '{e['text']}' | high-risk={e['is_high_risk']}")

Patching BC5CDR config for spaCy 3.8 compatibility...
  [WARN] config.cfg not found at /usr/local/lib/python3.12/dist-packages/en_ner_bc5cdr_md/config.cfg, skipping patch.
Loading scispaCy BC5CDR model...


/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


NER pipeline components: ['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'parser', 'ner']

Test entity extraction:
  [DISEASE] 'Diabetes Mellitus' | high-risk=True
  [CHEMICAL] 'Metformin' | high-risk=True


## Step 5: Transcript Chunker (Sliding Window)

The ASR transcript is split into overlapping 3-sentence windows.  
Each entity is scored against **all** chunks; only the best match is used.  
This ensures a drug mentioned early in the consultation can ground a claim appearing later in the generated note.

In [4]:
def chunk_transcript(transcript: str, chunk_size: int = 3) -> List[str]:
    """
    Sliding window chunker with overlap of (chunk_size - 1).
    chunk_size=3 means each chunk covers 3 consecutive sentences,
    sliding forward by 1 sentence per step.
    """
    sentences = re.split(r'(?<=[.!?])\s+', transcript.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

    if len(sentences) <= chunk_size:
        return [transcript]  # Short transcript -> single chunk

    chunks = []
    for i in range(0, len(sentences) - chunk_size + 1):
        chunks.append(" ".join(sentences[i : i + chunk_size]))

    # Ensure the final sentences are included
    last_chunk = " ".join(sentences[-chunk_size:])
    if last_chunk not in chunks:
        chunks.append(last_chunk)

    return chunks


# Test
# Test
test_tx = (
    'Patient reports severe headache for 3 days. '
    'Blood pressure is 145/90. Doctor prescribes Amlodipine 5mg. '
    'Follow up in 2 weeks. Patient mentions dizziness.'
)
chunks = chunk_transcript(test_tx)
print(f"Generated {len(chunks)} chunks:")
for i, c in enumerate(chunks):
    print(f"  [{i}] {c[:80]}...")

Generated 3 chunks:
  [0] Patient reports severe headache for 3 days. Blood pressure is 145/90. Doctor pre...
  [1] Blood pressure is 145/90. Doctor prescribes Amlodipine 5mg. Follow up in 2 weeks...
  [2] Doctor prescribes Amlodipine 5mg. Follow up in 2 weeks. Patient mentions dizzine...


## Step 6 — Stage 2: BiomedBERT + Cross-Encoder Grounding

**Two-pass scoring:**

| Pass | Model | Purpose |
|------|-------|---------|
| 1 (Fast) | BiomedBERT mean-pooled embeddings + cosine sim | Filter to top-3 transcript chunks |
| 2 (Precise) | Cross-Encoder (claim × chunk jointly) | Re-rank top-3 for final score |

**Final score = 0.4 × embedding score + 0.6 × cross-encoder score**  

Why Cross-Encoder for pass 2? Unlike bi-encoders that embed independently, cross-encoders process claim+evidence jointly — catching nuances like 'Metformin 500mg' ≠ 'Metformin 250mg' that cosine similarity misses.

In [5]:
print("Loading BiomedBERT for medical semantic embeddings...")
biomedbert_tokenizer = AutoTokenizer.from_pretrained(CONFIG['biomedbert_model'])
biomedbert_model = AutoModel.from_pretrained(CONFIG['biomedbert_model'])
biomedbert_model = biomedbert_model.to(CONFIG['device'])
biomedbert_model.eval()

print("Loading Cross-Encoder for claim-evidence re-ranking...")
cross_encoder = CrossEncoder(CONFIG['cross_encoder_model'], max_length=512)

print("Models loaded successfully!")

Loading BiomedBERT for medical semantic embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading Cross-Encoder for claim-evidence re-ranking...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models loaded successfully!


In [6]:
def get_biomedbert_embedding(text: str) -> np.ndarray:
    """
    Get mean-pooled BiomedBERT embedding.
    Mean pooling over all token embeddings outperforms CLS-only
    for semantic similarity (Reimers & Gurevych, 2019).
    """
    inputs = biomedbert_tokenizer(
        text, return_tensors='pt', truncation=True, max_length=512, padding=True
    ).to(CONFIG['device'])

    with torch.no_grad():
        outputs = biomedbert_model(**inputs)

    token_emb = outputs.last_hidden_state        # [1, seq_len, hidden]
    mask = inputs['attention_mask']               # [1, seq_len]
    mask_expanded = mask.unsqueeze(-1).expand(token_emb.size()).float()
    mean_pooled = torch.sum(token_emb * mask_expanded, 1) / \
                  torch.clamp(mask_expanded.sum(1), min=1e-9)
    return mean_pooled.squeeze().cpu().numpy()


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    dot = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    return float(dot / norm) if norm > 0 else 0.0


def ground_entity(
    entity_text: str,
    claim_sentence: str,
    transcript_chunks: List[str]
) -> Tuple[float, str]:
    """
    Two-pass grounding:
      Pass 1: BiomedBERT cosine similarity -> shortlist top-3 chunks
      Pass 2: Cross-Encoder re-ranks top-3
      Final:  0.4 * emb_score + 0.6 * ce_score

    Returns (best_score [0,1], best_matching_chunk)
    """
    query = f"{entity_text}. Context: {claim_sentence}"

    # --- Pass 1: embedding pre-filter ---
    query_emb = get_biomedbert_embedding(query)
    scored_chunks = []
    for chunk in transcript_chunks:
        c_emb = get_biomedbert_embedding(chunk)
        scored_chunks.append((cosine_sim(query_emb, c_emb), chunk))
    scored_chunks.sort(key=lambda x: x[0], reverse=True)
    top3 = [c for _, c in scored_chunks[:3]]

    if not top3:
        return (0.0, "")

    # --- Pass 2: cross-encoder re-ranking ---
    ce_scores_raw = cross_encoder.predict([(query, c) for c in top3])
    ce_scores_norm = 1.0 / (1.0 + np.exp(-ce_scores_raw))  # sigmoid to [0,1]
    emb_scores_top3 = [scored_chunks[i][0] for i in range(len(top3))]

    blended = [0.4 * emb_scores_top3[i] + 0.6 * ce_scores_norm[i]
               for i in range(len(top3))]

    best_idx = int(np.argmax(blended))
    return float(blended[best_idx]), top3[best_idx]


print("Grounding functions defined.")

Grounding functions defined.


## Step 7: Confidence Label Assignment

In [7]:
def assign_label(score: float, is_high_risk: bool) -> ConfidenceLabel:
    """
    HIGH-RISK entities (drugs, diagnoses) require a 5% higher score
    to earn VERIFIED status — reducing false-safe for hallucinated medications.
    """
    v_thresh = CONFIG['verified_threshold'] + (0.05 if is_high_risk else 0.0)
    u_thresh = CONFIG['uncertain_threshold']
    if score >= v_thresh:
        return ConfidenceLabel.VERIFIED
    elif score >= u_thresh:
        return ConfidenceLabel.UNCERTAIN
    else:
        return ConfidenceLabel.HALLUCINATED

high_risk_thresh = CONFIG['verified_threshold'] + 0.05
print(f"  High-risk : VERIFIED >= {high_risk_thresh:.2f} | "
      f"UNCERTAIN >= {CONFIG['uncertain_threshold']:.2f} | HALLUCINATED < {CONFIG['uncertain_threshold']:.2f}")


  High-risk : VERIFIED >= 0.77 | UNCERTAIN >= 0.45 | HALLUCINATED < 0.45


## Step 8: Full Hallucination Guard Pipeline

Ties Stage 1 (NER) and Stage 2 (BiomedBERT Grounding) together.

In [8]:
def run_hallucination_guard(
    transcript: str,
    generated_document: str,
    verbose: bool = True
) -> HallucinationReport:
    """
    Main pipeline:
      1. Extract clinical entities from generated_document (scispaCy)
      2. Chunk transcript into sliding windows
      3. Score each entity vs all transcript chunks (BiomedBERT + CE)
      4. Assign GREEN/YELLOW/RED labels
    """
    print("=" * 60)
    print("STAGE 1: Clinical NER (scispaCy BC5CDR)")
    print("=" * 60)
    raw_entities = extract_clinical_entities(generated_document)
    print(f"Found {len(raw_entities)} clinical entities.")

    if not raw_entities:
        print("[WARNING] No entities found. Check document.")
        return HallucinationReport(
            original_transcript=transcript,
            generated_document=generated_document,
            overall_trust_score=1.0
        )

    print("\n" + "=" * 60)
    print("STAGE 2: BiomedBERT + Cross-Encoder Grounding")
    print("=" * 60)
    transcript_chunks = chunk_transcript(transcript, CONFIG['chunk_size_sentences'])
    print(f"Transcript split into {len(transcript_chunks)} chunks.")

    clinical_entities = []
    flagged_sentences = set()

    for raw_ent in raw_entities:
        score, best_chunk = ground_entity(
            entity_text=raw_ent['text'],
            claim_sentence=raw_ent['sentence'],
            transcript_chunks=transcript_chunks
        )
        label = assign_label(score, raw_ent['is_high_risk'])

        entity = ClinicalEntity(
            text=raw_ent['text'],
            entity_type=raw_ent['label'],
            source_sentence=raw_ent['sentence'],
            confidence_score=round(score, 4),
            label=label,
            best_matching_transcript_chunk=best_chunk,
            is_high_risk=raw_ent['is_high_risk']
        )
        clinical_entities.append(entity)

        if label != ConfidenceLabel.VERIFIED:
            flagged_sentences.add(raw_ent['sentence'])

        if verbose:
            icons = {"GREEN": "✅", "YELLOW": "🟡", "RED": "🔴"}
            risk = "⚠ HIGH-RISK" if raw_ent["is_high_risk"] else ""
            print(f"  {icons[label.value]} [{label.value:12s}] "
                  f"{raw_ent['label']:10s} | '{raw_ent['text']}' "
                  f"| score={score:.3f} {risk}")

    trust = float(np.mean([e.confidence_score for e in clinical_entities]))

    return HallucinationReport(
        original_transcript=transcript,
        generated_document=generated_document,
        entities=clinical_entities,
        overall_trust_score=trust,
        flagged_sentences=list(flagged_sentences)
    )


print("Pipeline function ready.")

Pipeline function ready.


## Step 9: Test Case 1 — Grounded Document (expect mostly GREEN)

All entities in the generated note were actually spoken in the consultation.

In [9]:
transcript_grounded = """
Doctor: Good morning, how are you feeling today?
Patient: Not so good. I have had chest pain since yesterday evening.
Doctor: Where exactly, can you point to it?
Patient: Left side. Worse when I breathe in deeply.
Doctor: Any shortness of breath?
Patient: Mild shortness of breath. No cough.
Doctor: Blood pressure today is 150 over 95, that is elevated.
Doctor: ECG shows no acute changes.
Doctor: I am prescribing Amlodipine 5mg once daily for hypertension.
Doctor: Also take Aspirin 75mg once daily as precaution.
Patient: Any side effects?
Doctor: Amlodipine may cause ankle swelling. Come back in two weeks.
"""

document_grounded = """
Chief Complaint: Left-sided chest pain since yesterday evening, worsening on inspiration, with mild dyspnea.

Vitals: Blood pressure 150/95 mmHg (elevated).

Assessment: Hypertension. ECG shows no acute changes.

Plan:
1. Amlodipine 5mg once daily.
2. Aspirin 75mg once daily.
3. Follow-up in 2 weeks.
"""

print("Running guard on grounded document...")
report_grounded = run_hallucination_guard(transcript_grounded, document_grounded, verbose=True)

Running guard on grounded document...
STAGE 1: Clinical NER (scispaCy BC5CDR)
Found 5 clinical entities.

STAGE 2: BiomedBERT + Cross-Encoder Grounding
Transcript split into 13 chunks.
  ✅ [GREEN       ] DISEASE    | 'chest pain' | score=0.910 ⚠ HIGH-RISK
  🟡 [YELLOW      ] DISEASE    | 'dyspnea' | score=0.717 ⚠ HIGH-RISK
  🔴 [RED         ] DISEASE    | 'Hypertension' | score=0.384 ⚠ HIGH-RISK
  ✅ [GREEN       ] CHEMICAL   | 'Amlodipine' | score=0.989 ⚠ HIGH-RISK
  ✅ [GREEN       ] CHEMICAL   | 'Aspirin' | score=0.987 ⚠ HIGH-RISK


## Step 10: Test Case 2 — Hallucinated Document (expect RED for invented items)

**Injected hallucinations:**  
- `Type 2 Diabetes Mellitus` — never mentioned in the audio  
- `Atorvastatin 20mg` — never prescribed  
- `Metformin 500mg` — never prescribed  

The guard should flag these RED while keeping `Amlodipine` and `Aspirin` GREEN.

In [10]:
transcript_hallucinated = transcript_grounded  # Same audio, no diabetes mentioned

document_hallucinated = """
Chief Complaint: Left-sided chest pain since yesterday evening, worsening on inspiration, with mild dyspnea.

Vitals: Blood pressure 150/95 mmHg (elevated).

Assessment: Hypertension. Type 2 Diabetes Mellitus (known case). ECG shows no acute changes.

Plan:
1. Amlodipine 5mg once daily for blood pressure.
2. Aspirin 75mg once daily.
3. Atorvastatin 20mg at night for hyperlipidemia.
4. Metformin 500mg twice daily for diabetes management.
5. Follow-up in 2 weeks.
"""

print("Running guard on document with injected hallucinations...")
report_hallucinated = run_hallucination_guard(transcript_hallucinated, document_hallucinated, verbose=True)

Running guard on document with injected hallucinations...
STAGE 1: Clinical NER (scispaCy BC5CDR)
Found 10 clinical entities.

STAGE 2: BiomedBERT + Cross-Encoder Grounding
Transcript split into 13 chunks.
  ✅ [GREEN       ] DISEASE    | 'chest pain' | score=0.910 ⚠ HIGH-RISK
  🟡 [YELLOW      ] DISEASE    | 'dyspnea' | score=0.717 ⚠ HIGH-RISK
  🔴 [RED         ] DISEASE    | 'Hypertension' | score=0.384 ⚠ HIGH-RISK
  🔴 [RED         ] DISEASE    | 'Diabetes Mellitus' | score=0.389 ⚠ HIGH-RISK
  ✅ [GREEN       ] CHEMICAL   | 'Amlodipine' | score=0.991 ⚠ HIGH-RISK
  ✅ [GREEN       ] CHEMICAL   | 'Aspirin' | score=0.987 ⚠ HIGH-RISK
  🔴 [RED         ] CHEMICAL   | 'Atorvastatin' | score=0.393 ⚠ HIGH-RISK
  🔴 [RED         ] DISEASE    | 'hyperlipidemia' | score=0.392 ⚠ HIGH-RISK
  🔴 [RED         ] CHEMICAL   | 'Metformin' | score=0.394 ⚠ HIGH-RISK
  🔴 [RED         ] DISEASE    | 'diabetes' | score=0.395 ⚠ HIGH-RISK


## Step 11: Physician-Facing Report Generator

Generates the structured JSON report + inline-annotated document for the physician UI.

In [11]:
def generate_physician_report(report: HallucinationReport) -> Dict:
    """
    Converts HallucinationReport into physician-readable JSON.
    Includes overall trust score, critical warnings, per-entity scores,
    and an annotated document with inline [FLAG] markers.
    """
    ICONS = {
        "GREEN":  "[VERIFIED]",
        "YELLOW": "[CHECK]",
        "RED":    "[FLAG - NOT IN AUDIO]"
    }

    # Annotate document inline (longest entity first to avoid partial replacements)
    annotated_doc = report.generated_document
    for entity in sorted(report.entities, key=lambda e: -len(e.text)):
        tag = ICONS[entity.label.value]
        annotated_doc = annotated_doc.replace(
            entity.text, f"{entity.text} {tag}", 1
        )

    critical_warnings = [
        {
            "entity": e.text,
            "type": e.entity_type,
            "message": f"WARNING: '{e.text}' ({e.entity_type}) was NOT in the audio. Please verify before signing.",
            "confidence_score": e.confidence_score
        }
        for e in report.entities
        if e.label == ConfidenceLabel.HALLUCINATED and e.is_high_risk
    ]

    trust = report.overall_trust_score
    trust_band = 'HIGH' if trust >= 0.72 else ('MEDIUM' if trust >= 0.45 else 'LOW')

    return {
        "overall_trust_score": round(trust, 3),
        "trust_band": trust_band,
        "summary": report.summary(),
        "critical_warnings": critical_warnings,
        "entity_annotations": [
            {
                "entity": e.text,
                "type": e.entity_type,
                "label": e.label.value,
                "score": e.confidence_score,
                "is_high_risk": e.is_high_risk,
                "supporting_evidence": e.best_matching_transcript_chunk
            }
            for e in report.entities
        ],
        "annotated_document": annotated_doc,
        "flagged_sentences": report.flagged_sentences
    }


print("\n" + "="*70)
print("REPORT: GROUNDED DOCUMENT")
print("="*70)
r1 = generate_physician_report(report_grounded)
print(json.dumps(r1['summary'], indent=2))
print(f"Trust Band: {r1['trust_band']}")

print("\n" + "="*70)
print("REPORT: HALLUCINATED DOCUMENT")
print("="*70)
r2 = generate_physician_report(report_hallucinated)
print(json.dumps(r2['summary'], indent=2))
print(f"Trust Band: {r2['trust_band']}")
print("\nCritical Warnings:")
for w in r2['critical_warnings']:
    print(f"  ALERT: {w['message']}")


REPORT: GROUNDED DOCUMENT
{
  "total_entities": 5,
  "verified": 3,
  "uncertain": 1,
  "hallucinated": 1,
  "trust_score": 0.797,
  "high_risk_hallucinations": [
    "Hypertension"
  ]
}
Trust Band: HIGH

REPORT: HALLUCINATED DOCUMENT
{
  "total_entities": 10,
  "verified": 3,
  "uncertain": 1,
  "hallucinated": 6,
  "trust_score": 0.595,
  "high_risk_hallucinations": [
    "Hypertension",
    "Diabetes Mellitus",
    "Atorvastatin",
    "hyperlipidemia",
    "Metformin",
    "diabetes"
  ]
}
Trust Band: MEDIUM

Critical Warnings:
  ALERT: WARNING: 'Hypertension' (DISEASE) was NOT in the audio. Please verify before signing.
  ALERT: WARNING: 'Diabetes Mellitus' (DISEASE) was NOT in the audio. Please verify before signing.
  ALERT: WARNING: 'Atorvastatin' (CHEMICAL) was NOT in the audio. Please verify before signing.
  ALERT: WARNING: 'hyperlipidemia' (DISEASE) was NOT in the audio. Please verify before signing.
  ALERT: WARNING: 'Metformin' (CHEMICAL) was NOT in the audio. Please ver

## Step 12: Annotated Document Output

In [12]:
print("--- ANNOTATED DOCUMENT (Hallucinated version) ---\n")
print(r2['annotated_document'])
print("\n--- LEGEND ---")
print("[VERIFIED]            = Grounded in audio (safe)")
print("[CHECK]               = Weakly grounded - doctor should verify")
print("[FLAG - NOT IN AUDIO] = Not found in audio - likely hallucinated")

--- ANNOTATED DOCUMENT (Hallucinated version) ---


Chief Complaint: Left-sided chest pain [VERIFIED] since yesterday evening, worsening on inspiration, with mild dyspnea [CHECK].

Vitals: Blood pressure 150/95 mmHg (elevated).

Assessment: Hypertension [FLAG - NOT IN AUDIO]. Type 2 Diabetes Mellitus [FLAG - NOT IN AUDIO] (known case). ECG shows no acute changes.

Plan:
1. Amlodipine [VERIFIED] 5mg once daily for blood pressure.
2. Aspirin [VERIFIED] 75mg once daily.
3. Atorvastatin [FLAG - NOT IN AUDIO] 20mg at night for hyperlipidemia [FLAG - NOT IN AUDIO].
4. Metformin [FLAG - NOT IN AUDIO] 500mg twice daily for diabetes [FLAG - NOT IN AUDIO] management.
5. Follow-up in 2 weeks.


--- LEGEND ---
[VERIFIED]            = Grounded in audio (safe)
[CHECK]               = Weakly grounded - doctor should verify
[FLAG - NOT IN AUDIO] = Not found in audio - likely hallucinated


## Step 13: Integration API — `HallucinationGuard` Class

Clean API to integrate with the main `ClinicASR-IND` pipeline from your other notebooks.

In [13]:
class HallucinationGuard:
    """
    Production API for the two-stage hallucination guard.

    Usage in main pipeline:
    -------
    guard = HallucinationGuard()

    # After Whisper transcription + LLM note generation:
    result = guard.check(whisper_transcript, llm_generated_note)

    # result keys:
    #   result['trust_band']         -> 'HIGH' / 'MEDIUM' / 'LOW'
    #   result['critical_warnings']  -> flagged drugs/diagnoses
    #   result['annotated_document'] -> note with inline [FLAG] markers
    #   result['entity_annotations'] -> per-entity scores
    """

    def __init__(self):
        print("HallucinationGuard initialized.")
        print("  Stage 1: scispaCy en_ner_bc5cdr_md")
        print("  Stage 2: BiomedBERT embeddings + CrossEncoder re-ranking")
        print(f"  Device: {CONFIG['device']}")

    def check(
        self,
        whisper_transcript: str,
        generated_note: str,
        verbose: bool = False
    ) -> Dict:
        """Run the full hallucination guard. Returns physician report dict."""
        report = run_hallucination_guard(whisper_transcript, generated_note, verbose=verbose)
        return generate_physician_report(report)

    def is_safe_to_sign(self, physician_report: Dict) -> bool:
        """
        Returns True only if no high-risk hallucinations were found.
        Use as a gate before auto-filing the note.
        """
        return len(physician_report['critical_warnings']) == 0


# Demo
guard = HallucinationGuard()
result = guard.check(transcript_grounded, document_hallucinated, verbose=False)

print(f"Trust Score : {result['overall_trust_score']}")
print(f"Trust Band  : {result['trust_band']}")
print(f"Safe to sign: {guard.is_safe_to_sign(result)}")
print("\nCritical hallucinations found:")
for w in result['critical_warnings']:
    print(f"  -> {w['entity']} ({w['type']}) score={w['confidence_score']:.3f}")

HallucinationGuard initialized.
  Stage 1: scispaCy en_ner_bc5cdr_md
  Stage 2: BiomedBERT embeddings + CrossEncoder re-ranking
  Device: cpu
STAGE 1: Clinical NER (scispaCy BC5CDR)
Found 10 clinical entities.

STAGE 2: BiomedBERT + Cross-Encoder Grounding
Transcript split into 13 chunks.
Trust Score : 0.595
Trust Band  : MEDIUM
Safe to sign: False

Critical hallucinations found:
  -> Hypertension (DISEASE) score=0.384
  -> Diabetes Mellitus (DISEASE) score=0.389
  -> Atorvastatin (CHEMICAL) score=0.393
  -> hyperlipidemia (DISEASE) score=0.392
  -> Metformin (CHEMICAL) score=0.394
  -> diabetes (DISEASE) score=0.395


## Step 14: Save Report to JSON

In [14]:
import os

# Adjust path as needed for your Google Drive setup
output_dir  = '/content/drive/MyDrive/my_asr_project'
output_path = os.path.join(output_dir, 'hallucination_report_sample.json')
os.makedirs(output_dir, exist_ok=True)

with open(output_path, 'w') as f:
    json.dump(result, f, indent=2)

print(f'Report saved to: {output_path}')
print('This JSON is consumed by the physician UI to render the annotated document.')

Report saved to: /content/drive/MyDrive/my_asr_project/hallucination_report_sample.json
This JSON is consumed by the physician UI to render the annotated document.


# Running it on the dataset

In [33]:
!unzip Medical_Audio_Dataset.zip

Archive:  Medical_Audio_Dataset.zip
   creating: Medical_Audio_Dataset/
  inflating: Medical_Audio_Dataset/sample_0.wav  
  inflating: Medical_Audio_Dataset/sample_1.wav  
  inflating: Medical_Audio_Dataset/sample_2.wav  
  inflating: Medical_Audio_Dataset/sample_3.wav  
  inflating: Medical_Audio_Dataset/sample_4.wav  
  inflating: Medical_Audio_Dataset/sample_5.wav  
  inflating: Medical_Audio_Dataset/sample_6.wav  
  inflating: Medical_Audio_Dataset/sample_7.wav  
  inflating: Medical_Audio_Dataset/sample_8.wav  
  inflating: Medical_Audio_Dataset/sample_9.wav  
  inflating: Medical_Audio_Dataset/sample_10.wav  
  inflating: Medical_Audio_Dataset/sample_11.wav  
  inflating: Medical_Audio_Dataset/sample_12.wav  
  inflating: Medical_Audio_Dataset/sample_13.wav  
  inflating: Medical_Audio_Dataset/sample_14.wav  
  inflating: Medical_Audio_Dataset/sample_15.wav  
  inflating: Medical_Audio_Dataset/sample_16.wav  
  inflating: Medical_Audio_Dataset/sample_17.wav  
  inflating: Medical

In [34]:
!unzip medical_whisper_final\ \(1\).zip -d /content/medical_whisper_final/

Archive:  medical_whisper_final (1).zip
   creating: /content/medical_whisper_final/medical_whisper_final/
  inflating: /content/medical_whisper_final/medical_whisper_final/README.md  
  inflating: /content/medical_whisper_final/medical_whisper_final/adapter_model.safetensors  
  inflating: /content/medical_whisper_final/medical_whisper_final/adapter_config.json  
  inflating: /content/medical_whisper_final/medical_whisper_final/tokenizer_config.json  
  inflating: /content/medical_whisper_final/medical_whisper_final/tokenizer.json  
  inflating: /content/medical_whisper_final/medical_whisper_final/processor_config.json  
  inflating: /content/medical_whisper_final/medical_whisper_final/training_args.bin  
   creating: /content/medical_whisper_final/medical_whisper_final/.ipynb_checkpoints/
  inflating: /content/medical_whisper_final/medical_whisper_final/.ipynb_checkpoints/adapter_config-checkpoint.json  


In [2]:
!pip install -q "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 17.7 MB/s eta 0:00:00


In [15]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from peft import PeftModel, PeftConfig
import torch

MODEL_PATH = "/content/medical_whisper_final/medical_whisper_final"  # or full Drive path if you moved it

# Load processor
processor = WhisperProcessor.from_pretrained(MODEL_PATH)

# Load base Whisper + your LoRA weights on top
base_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
model = PeftModel.from_pretrained(base_model, MODEL_PATH)
model.eval()
model = model.to("cuda" if torch.cuda.is_available() else "cpu")

print("✅ Model loaded successfully")

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

✅ Model loaded successfully


In [16]:
import librosa

def transcribe_wav(wav_path):
    # Load audio and resample to 16kHz (Whisper requirement)
    audio, sr = librosa.load(wav_path, sr=16000)

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.to(model.device)

    with torch.no_grad():
        predicted_ids = model.generate(inputs)

    transcript = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return transcript

In [19]:
import glob, os
import pandas as pd

AUDIO_DIR = "/content/Medical_Audio_Dataset"   # <-- update this
wav_files = glob.glob(os.path.join(AUDIO_DIR, "*.wav"))
print(f"Found {len(wav_files)} files")

guard = HallucinationGuard()
results = []

for wav_path in wav_files:
    print(f"\nProcessing: {os.path.basename(wav_path)}")

    # Transcribe with your fine-tuned model
    transcript = transcribe_wav(wav_path)
    print(f"  Transcript: {transcript[:80]}...")

    # Run hallucination guard
    report = guard.check(transcript, transcript, verbose=False)
    # NOTE: since your model does transcription (not note generation yet),
    # both inputs are the transcript for now.
    # When you add a note-generation LLM later, replace the second
    # argument with the generated note.

    results.append({
        "file": os.path.basename(wav_path),
        "transcript": transcript,
        "trust_score": report["overall_trust_score"],
        "trust_band": report["trust_band"],
        "hallucinated_entities": report["summary"]["high_risk_hallucinations"],
        "safe_to_sign": guard.is_safe_to_sign(report)
    })

df = pd.DataFrame(results)
df.to_csv("hallucination_results.csv", index=False)
print("\n✅ Done!")
print(df[["file", "trust_band", "safe_to_sign"]])

Found 20 files
HallucinationGuard initialized.
  Stage 1: scispaCy en_ner_bc5cdr_md
  Stage 2: BiomedBERT embeddings + CrossEncoder re-ranking
  Device: cpu

Processing: sample_6.wav


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log

  Transcript: Doctor, tumhare knee ka pain kaisa hai? Patient, better hai, but stairs chadhte ...
STAGE 1: Clinical NER (scispaCy BC5CDR)
Found 1 clinical entities.

STAGE 2: BiomedBERT + Cross-Encoder Grounding
Transcript split into 1 chunks.

Processing: sample_18.wav
  Transcript: Doctor, aaj aapka blood pressure kaafi better hai. Patient, mujhe lagta hai diet...
STAGE 1: Clinical NER (scispaCy BC5CDR)
Found 0 clinical entities.
[WARNING] No entities found. Check document.

Processing: sample_16.wav
  Transcript: Doctor, kya new medication se aapko nausea hua hai? Patient, thoda sa, but kuch ...
STAGE 1: Clinical NER (scispaCy BC5CDR)
Found 1 clinical entities.

STAGE 2: BiomedBERT + Cross-Encoder Grounding
Transcript split into 1 chunks.

Processing: sample_14.wav
  Transcript: Doctor, aapke iron levels kaafi low hain. Patient, kya isi wajah se main itna ti...
STAGE 1: Clinical NER (scispaCy BC5CDR)
Found 2 clinical entities.

STAGE 2: BiomedBERT + Cross-Encoder Grounding
Transcrip

In [17]:
medical_scripts = [
    "Doctor: Hello, how are you feeling? Patient: Mujhe teen din se bahut tez headache ho raha hai. Doctor: Theek hai, chaliye aapka blood pressure check karte hain; yeh thoda high hai.",
    "Patient: Doctor, aaj mera back pain aur badh gaya hai. Doctor: Kya aap Ibuprofen le rahe the jo maine prescribe ki thi? Patient: Sirf ek baar, usse mera stomach thoda uneasy feel hua.",
    "Doctor: Aapka blood sugar level thoda elevated hai. Patient: Kya mujhe Metformin phir se start karni chahiye? Doctor: Haan, please din mein do baar 500 milligrams lijiye.",
    "Patient: Mujhe lagatar cough aur fever hai. Doctor: Chaliye, main aapke lungs sunta hoon. Yeh viral infection jaisa lag raha hai, plenty of fluids piyen.",
    "Doctor: Report mein aapka cholesterol high dikh raha hai. Patient: Kya mujhe apni diet change karni padegi? Doctor: Haan, oily foods avoid kijiye aur Atorvastatin continue kijiye.",
    "Patient: Meri skin lately bahut itchy ho rahi hai. Doctor: Yeh allergic reaction lag raha hai, main Cetirizine naam ka antihistamine prescribe karunga.",
    "Doctor: Tumhare knee ka pain kaisa hai? Patient: Better hai, but stairs chadhte time abhi bhi dard hota hai. Doctor: Jo ointment prescribe ki thi, uska use continue karo.",
    "Patient: Mujhe khade hote hi bahut chakkar aate hain. Doctor: Yeh tumhari current medication ki wajah se ho sakta hai, chalo Amlodipine ki dosage adjust karte hain.",
    "Doctor: Kya tumne antibiotics ka course complete kar liya? Patient: Haan, mere gale ki swelling kaafi kam ho gayi hai.",
    "Patient: Raat ko mujhe bahut anxiety feel hoti hai. Doctor: Chalo do hafton ke liye Escitalopram jaisi mild sedative try karte hain.",
    "Doctor: Aaj tumhare thyroid levels thode off hain. Patient: Kya mujhe thyroxine ki dose badha leni chahiye? Doctor: Haan, kal se seventy-five micrograms lena.",
    "Patient: Subah se meri aankhon se bahut paani aa raha hai. Doctor: Yeh seasonal allergies mein common hai, in eye drops ko din mein teen baar use karo.",
    "Doctor: Aap apne diabetes ko kaise manage kar rahe ho? Patient: Main zyada walk karne ki koshish kar raha hoon, but mere pair abhi bhi numb feel hote hain. Doctor: Humein aapki nerve health ko aur closely monitor karna hoga.",
    "Patient: Mere chest mein sharp pain ho raha hai. Doctor: Please baith jao; mujhe cardiac issues rule out karne ke liye immediately EKG karna hoga.",
    "Doctor: Aapke iron levels kaafi low hain. Patient: Kya isi wajah se main itna tired feel karta hoon? Doctor: Haan, lunch ke baad daily yeh iron supplements lo.",
    "Patient: Mere arm par rash hai. Doctor: Yeh contact dermatitis jaisa lag raha hai, is hydrocortisone cream ko din mein do baar lagao.",
    "Doctor: Kya new medication se aapko nausea hua hai? Patient: Thoda sa, but kuch khaane ke baad theek ho jaata hai.",
    "Patient: Mera bachcha kal raat se vomiting kar raha hai. Doctor: Chalo dehydration check karte hain; unhe thoda-thoda karke ORS do.",
    "Doctor: Aaj aapka blood pressure kaafi better hai. Patient: Mujhe lagta hai diet mein change se kaafi help hui. Doctor: Excellent, aise hi continue rakho.",
    "Patient: Mujhe is persistent cough ko lekar tension ho rahi hai. Doctor: Chalo chest X-ray karwa lete hain taaki confirm ho jaye ki lungs mein congestion to nahi hai."
]

In [18]:
import glob, os
import pandas as pd

AUDIO_DIR = "/content/Medical_Audio_Dataset"

# Numeric sort to ensure sample_0 -> script[0], sample_1 -> script[1], etc.
wav_files = sorted(
    glob.glob(os.path.join(AUDIO_DIR, "*.wav")),
    key=lambda x: int(os.path.basename(x).replace("sample_", "").replace(".wav", ""))
)
print(f"Found {len(wav_files)} files")

# Sanity check pairing
print("\nVerifying pairing:")
for i, (wav, script) in enumerate(zip(wav_files, medical_scripts)):
    print(f"  {os.path.basename(wav)} -> {script[:60]}...")

assert len(wav_files) == len(medical_scripts), \
    f"Mismatch: {len(wav_files)} wav files but {len(medical_scripts)} scripts"

Found 20 files

Verifying pairing:
  sample_0.wav -> Doctor: Hello, how are you feeling? Patient: Mujhe teen din ...
  sample_1.wav -> Patient: Doctor, aaj mera back pain aur badh gaya hai. Docto...
  sample_2.wav -> Doctor: Aapka blood sugar level thoda elevated hai. Patient:...
  sample_3.wav -> Patient: Mujhe lagatar cough aur fever hai. Doctor: Chaliye,...
  sample_4.wav -> Doctor: Report mein aapka cholesterol high dikh raha hai. Pa...
  sample_5.wav -> Patient: Meri skin lately bahut itchy ho rahi hai. Doctor: Y...
  sample_6.wav -> Doctor: Tumhare knee ka pain kaisa hai? Patient: Better hai,...
  sample_7.wav -> Patient: Mujhe khade hote hi bahut chakkar aate hain. Doctor...
  sample_8.wav -> Doctor: Kya tumne antibiotics ka course complete kar liya? P...
  sample_9.wav -> Patient: Raat ko mujhe bahut anxiety feel hoti hai. Doctor: ...
  sample_10.wav -> Doctor: Aaj tumhare thyroid levels thode off hain. Patient: ...
  sample_11.wav -> Patient: Subah se meri aankhon se bahut paa

In [19]:
guard = HallucinationGuard()
results = []

for i, (wav_path, original_script) in enumerate(zip(wav_files, medical_scripts)):
    print(f"\n[{i}] Processing: {os.path.basename(wav_path)}")

    # Step 1: Your fine-tuned Whisper transcribes the audio
    transcription = transcribe_wav(wav_path)
    print(f"  Original   : {original_script[:80]}...")
    print(f"  Transcribed: {transcription[:80]}...")

    # Step 2: Hallucination guard
    # original_script = ground truth (what was actually said)
    # transcription   = what your model produced
    report = guard.check(
        original_script,
        transcription,
        verbose=False
    )

    results.append({
        "file": os.path.basename(wav_path),
        "original_script": original_script,
        "transcription": transcription,
        "annotated_transcription": report["annotated_document"],
        "trust_score": report["overall_trust_score"],
        "trust_band": report["trust_band"],
        "verified_entities": report["summary"]["verified"],
        "uncertain_entities": report["summary"]["uncertain"],
        "hallucinated_entities": report["summary"]["hallucinated"],
        "high_risk_hallucinations": report["summary"]["high_risk_hallucinations"],
        "transcription_accurate": guard.is_safe_to_sign(report)
    })
    print(f"  Trust: {report['trust_band']} | "
          f"Accurate: {guard.is_safe_to_sign(report)} | "
          f"Errors: {report['summary']['high_risk_hallucinations']}")

# Save results
df = pd.DataFrame(results)
df.to_csv("transcription_hallucination_results.csv", index=False)

# Summary
print("\n" + "="*50)
print("TRANSCRIPTION QUALITY SUMMARY")
print("="*50)
print(f"  HIGH trust    : {(df['trust_band']=='HIGH').sum()}/{len(wav_files)}")
print(f"  MEDIUM trust  : {(df['trust_band']=='MEDIUM').sum()}/{len(wav_files)}")
print(f"  LOW trust     : {(df['trust_band']=='LOW').sum()}/{len(wav_files)}")
print(f"  Fully accurate: {df['transcription_accurate'].sum()}/{len(wav_files)}")
print(f"\nFiles with transcription errors:")
flagged = df[~df['transcription_accurate']][["file", "trust_band", "high_risk_hallucinations"]]
print(flagged if len(flagged) > 0 else "  None — model transcribed everything correctly!")
print(f"\nResults saved to transcription_hallucination_results.csv")

HallucinationGuard initialized.
  Stage 1: scispaCy en_ner_bc5cdr_md
  Stage 2: BiomedBERT embeddings + CrossEncoder re-ranking
  Device: cpu

[0] Processing: sample_0.wav


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log

  Original   : Doctor: Hello, how are you feeling? Patient: Mujhe teen din se bahut tez headach...
  Transcribed: Doctor Hello, how are you feeling? Patient, mujhe teen din se bahut tez headache...
STAGE 1: Clinical NER (scispaCy BC5CDR)
Found 1 clinical entities.

STAGE 2: BiomedBERT + Cross-Encoder Grounding
Transcript split into 1 chunks.
  Trust: HIGH | Accurate: True | Errors: []

[1] Processing: sample_1.wav
  Original   : Patient: Doctor, aaj mera back pain aur badh gaya hai. Doctor: Kya aap Ibuprofen...
  Transcribed: Patienth Doctor, aaj mera back pain aur badh gaya hai. Doctor, kya aap Ibuprofen...
STAGE 1: Clinical NER (scispaCy BC5CDR)
Found 1 clinical entities.

STAGE 2: BiomedBERT + Cross-Encoder Grounding
Transcript split into 1 chunks.
  Trust: HIGH | Accurate: True | Errors: []

[2] Processing: sample_2.wav
  Original   : Doctor: Aapka blood sugar level thoda elevated hai. Patient: Kya mujhe Metformin...
  Transcribed: Doctor, aapka blood sugar level thoda elevated hai

In [20]:
from google.colab import files
files.download('transcription_hallucination_results.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>